# Make Plate-Resolved Tensors For Drugs and Pseudobulk Data

Mirrors `3_Make_tensors.ipynb` but keeps `plate` as a grouping dimension on the treatment side so the bundle is plate-aware. DMSO baselines stay per cell line (aggregating across plates with `n_cells_used` weights, same as notebook 3).

Outputs land in `data/Tahoe100M_tensor_artifacts_L1000_plate/` and the bundle adds:
- `plates`: list[str] aligned with `condition_keys`
- `plate_to_index`: dict over the full bundle
- `n_cells_used`: int64 tensor aligned with `condition_keys` (no longer summed over plates)

Used downstream by `8_adae_plate.ipynb` to train an AD-AE with the adversary attacking the fused latent against plate identity.

## Paths and Imports

In [ ]:
import ast
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch
from IPython.display import display

save_tensor_artifacts = True
LANDMARK_FEATURE_SPACE = "l1000_landmark_intersection"


def resolve_existing_path(path_like):
    path = Path(path_like)
    if path.is_absolute() and path.exists():
        return path
    for base_path in [Path.cwd(), *Path.cwd().parents]:
        candidate = base_path / path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {path} from {Path.cwd()}")


processed_data_path = resolve_existing_path("data/Tahoe100M_Pseudobulk_processed")
fingerprints_path = resolve_existing_path("data/Morganfingerprints.csv")
landmark_genes_path = resolve_existing_path("data/landmark_genes.csv")

tensor_artifacts_dir = processed_data_path.parent / "Tahoe100M_tensor_artifacts_L1000_plate"

h5ad_files = sorted(processed_data_path.glob("*.h5ad"))
if not h5ad_files:
    raise FileNotFoundError(f"No .h5ad files in {processed_data_path}")

print(
    {
        "processed_data_path": str(processed_data_path),
        "fingerprints_path": str(fingerprints_path),
        "landmark_genes_path": str(landmark_genes_path),
        "tensor_artifacts_dir": str(tensor_artifacts_dir),
        "n_h5ad_files": len(h5ad_files),
    }
)

## Shared Helpers (mirror `3_Make_tensors.ipynb`)

In [ ]:
def normalize_ensembl_id(gene_id):
    return str(gene_id).strip().split(".")[0]


def parse_condition_string(raw_condition):
    parsed = ast.literal_eval(raw_condition)
    if len(parsed) != 1:
        raise ValueError(f"Expected one drug/concentration entry, found {parsed!r}")
    drug, concentration, unit = parsed[0]
    return str(drug).strip(), float(concentration), str(unit).strip()


def normalize_condition_key_with_plate(cell_line, drug, concentration, concentration_unit, plate):
    base = "|||".join([
        str(cell_line),
        str(drug),
        format(float(concentration), ".15g"),
        str(concentration_unit),
    ])
    return f"{base}|plate={plate}"


def weighted_average_expression(expression_matrix, weights):
    weights = np.asarray(weights, dtype=np.float64)
    if weights.ndim != 1:
        raise ValueError("Weights must be one-dimensional.")
    total_weight = float(weights.sum())
    if total_weight <= 0:
        raise ValueError("Weights must sum to a positive value.")
    weighted = expression_matrix.T.dot(weights) / total_weight
    return np.asarray(weighted, dtype=np.float32).ravel()


def is_valid_fingerprint(bitstring, n_bits=2048):
    return isinstance(bitstring, str) and len(bitstring) == n_bits and set(bitstring) <= {"0", "1"}


def fingerprint_to_vector(bitstring, n_bits=2048):
    if not is_valid_fingerprint(bitstring, n_bits=n_bits):
        raise ValueError("Morgan fingerprint bitstring is missing or invalid.")
    return np.fromiter((float(bit) for bit in bitstring), dtype=np.float32, count=n_bits)


def is_all_zero_expression(vector):
    return bool(np.isclose(np.asarray(vector), 0.0).all())

## Load Morgan Fingerprints

In [ ]:
morgan_fingerprints = pd.read_csv(fingerprints_path, index_col=0)
morgan_fingerprints.index = morgan_fingerprints.index.astype(str).str.strip()
morgan_fingerprints["has_fingerprint"] = morgan_fingerprints["morgan_fingerprint"].apply(
    lambda bitstring: isinstance(bitstring, str) and len(bitstring) == 2048 and set(bitstring) <= {"0", "1"}
)
morgan_fingerprint_metadata_df = morgan_fingerprints.rename_axis("drug").reset_index()
missing_fingerprint_drugs = set(
    morgan_fingerprint_metadata_df.loc[~morgan_fingerprint_metadata_df["has_fingerprint"], "drug"]
)

print(
    f"Loaded {len(morgan_fingerprints)} Morgan fingerprint rows; "
    f"{len(missing_fingerprint_drugs)} drugs will be dropped from the treatment bundle."
)

## Validate Gene Space and Resolve Landmark Indices

In [ ]:
reference_gene_ids = None
for path in h5ad_files:
    backed = ad.read_h5ad(path, backed="r")
    current_gene_ids = [normalize_ensembl_id(g) for g in backed.var_names.to_list()]
    if reference_gene_ids is None:
        reference_gene_ids = current_gene_ids
    elif current_gene_ids != reference_gene_ids:
        backed.file.close()
        raise ValueError(f"Gene order mismatch in {path.name}.")
    backed.file.close()

gene_ids = [str(g) for g in reference_gene_ids]
gene_id_to_index = {g: i for i, g in enumerate(gene_ids)}

landmark_genes_df = pd.read_csv(landmark_genes_path)
landmark_genes_df["ensembl_id"] = landmark_genes_df["ensembl_id"].map(normalize_ensembl_id)
if landmark_genes_df["ensembl_id"].duplicated().any():
    raise ValueError("Duplicate landmark Ensembl IDs found.")

requested_landmark_gene_ids = landmark_genes_df["ensembl_id"].tolist()
requested_landmark_set = set(requested_landmark_gene_ids)
available_landmark_gene_indices = [i for i, g in enumerate(gene_ids) if g in requested_landmark_set]
available_landmark_gene_ids = [gene_ids[i] for i in available_landmark_gene_indices]
missing_landmark_gene_ids = [g for g in requested_landmark_gene_ids if g not in gene_id_to_index]
available_landmark_gene_indices_array = np.asarray(available_landmark_gene_indices, dtype=np.int64)

print(
    f"Validated {len(h5ad_files)} files share the same {len(gene_ids)} genes. "
    f"Resolved {len(available_landmark_gene_ids)} landmark genes; {len(missing_landmark_gene_ids)} missing."
)

## Aggregate DMSO Baselines (cell_line) and Treatment Expressions (cell_line, drug, conc, unit, **plate**)

DMSO is collapsed per cell line, weighted by `n_cells_used` — identical to `3_Make_tensors.ipynb`. Treatments are grouped with **plate as an extra key**, so the resulting bundle preserves plate identity per row.

In [ ]:
dmso_baseline_rows = []
dmso_baseline_vectors = []
treatment_rows = []
treatment_full_vectors = []

for path in h5ad_files:
    cell_line = path.stem
    adata = ad.read_h5ad(path)

    obs_df = adata.obs.copy()
    obs_df["row_index"] = np.arange(adata.n_obs)

    parsed = [parse_condition_string(s) for s in obs_df["drugname_drugconc"].tolist()]
    parsed_df = pd.DataFrame(parsed, columns=["parsed_drug", "concentration", "concentration_unit"], index=obs_df.index)
    obs_df["drug"] = obs_df["drug"].astype(str).str.strip()
    obs_df["plate"] = obs_df["plate"].astype(str).str.strip()
    if not parsed_df["parsed_drug"].eq(obs_df["drug"]).all():
        raise ValueError(f"Parsed drug names did not match `obs['drug']` for {path.name}.")
    obs_df = pd.concat([obs_df, parsed_df], axis=1)
    obs_df["cell_line"] = cell_line

    dmso_obs = obs_df.loc[obs_df["drug"].eq("DMSO_TF")]
    if dmso_obs.empty:
        raise ValueError(f"No DMSO_TF rows in {path.name}.")
    dmso_vector = weighted_average_expression(
        adata.X[dmso_obs["row_index"].to_numpy()],
        dmso_obs["n_cells_used"].to_numpy(dtype=np.float64),
    )
    if is_all_zero_expression(dmso_vector):
        raise ValueError(f"DMSO baseline is all-zero for {cell_line}.")
    dmso_baseline_vectors.append(dmso_vector)
    dmso_baseline_rows.append(
        {
            "cell_line": cell_line,
            "file_name": path.name,
            "source_count": int(dmso_obs.shape[0]),
            "total_n_cells_used": int(dmso_obs["n_cells_used"].sum()),
            "plates": ",".join(sorted(dmso_obs["plate"].unique().tolist())),
        }
    )

    treat_obs = obs_df.loc[~obs_df["drug"].eq("DMSO_TF")]
    treat_obs = treat_obs.loc[~treat_obs["drug"].isin(missing_fingerprint_drugs)]

    grouped = treat_obs.groupby(
        ["drug", "concentration", "concentration_unit", "plate"],
        sort=True,
        dropna=False,
        observed=True,
    )
    for (drug_name, concentration, concentration_unit, plate), group_df in grouped:
        treatment_full_vector = weighted_average_expression(
            adata.X[group_df["row_index"].to_numpy()],
            group_df["n_cells_used"].to_numpy(dtype=np.float64),
        )
        if is_all_zero_expression(treatment_full_vector):
            continue
        condition_key = normalize_condition_key_with_plate(
            cell_line, drug_name, concentration, concentration_unit, plate
        )
        treatment_full_vectors.append(treatment_full_vector)
        treatment_rows.append(
            {
                "condition_key": condition_key,
                "cell_line": cell_line,
                "file_name": path.name,
                "drug": drug_name,
                "concentration": float(concentration),
                "concentration_unit": concentration_unit,
                "plate": plate,
                "source_count": int(group_df.shape[0]),
                "n_cells_used": int(group_df["n_cells_used"].sum()),
            }
        )

    print(
        f"Aggregated {cell_line}: {dmso_obs.shape[0]} DMSO rows -> 1 baseline; "
        f"retained {grouped.ngroups} plate-resolved treatment profiles."
    )

treatment_df = pd.DataFrame(treatment_rows)
if treatment_df.empty:
    raise RuntimeError("Built zero treatment rows; check the raw pseudobulk inputs.")
if treatment_df["condition_key"].duplicated().any():
    duplicated = treatment_df.loc[treatment_df["condition_key"].duplicated(), "condition_key"].tolist()
    raise ValueError(f"Duplicate plate-resolved condition keys: {duplicated[:5]}")

dmso_baseline_df = pd.DataFrame(dmso_baseline_rows)
print(
    f"Built {len(dmso_baseline_df)} DMSO baselines and {len(treatment_df)} plate-resolved treatment profiles."
)
display(treatment_df.head())

## Build Tensor Bundles

In [ ]:
dmso_order = np.argsort(dmso_baseline_df["cell_line"].to_numpy(), kind="stable")
dmso_baseline_df = dmso_baseline_df.iloc[dmso_order].reset_index(drop=True)
dmso_baseline_matrix = np.vstack([dmso_baseline_vectors[i] for i in dmso_order]).astype(np.float32)
dmso_baseline_expression_tensor = torch.from_numpy(dmso_baseline_matrix)
dmso_cell_line_to_index = {cl: i for i, cl in enumerate(dmso_baseline_df["cell_line"])}

sort_keys = (
    treatment_df["cell_line"].astype(str)
    + "|" + treatment_df["drug"].astype(str)
    + "|" + treatment_df["concentration"].map(lambda v: format(float(v), ".15g"))
    + "|" + treatment_df["concentration_unit"].astype(str)
    + "|" + treatment_df["plate"].astype(str)
)
treatment_order = np.argsort(sort_keys.to_numpy(), kind="stable")
treatment_df = treatment_df.iloc[treatment_order].reset_index(drop=True)

treatment_full_matrix = np.vstack([treatment_full_vectors[i] for i in treatment_order]).astype(np.float32)
treatment_landmark_matrix = treatment_full_matrix[:, available_landmark_gene_indices_array].astype(np.float32, copy=False)
treatment_expression_tensor = torch.from_numpy(treatment_landmark_matrix)
treatment_landmark_gene_ids = available_landmark_gene_ids.copy()

treatment_condition_key_to_index = {
    key: idx for idx, key in enumerate(treatment_df["condition_key"])
}
plate_to_index = {plate: idx for idx, plate in enumerate(sorted(treatment_df["plate"].unique().tolist()))}

morgan_fingerprint_df = morgan_fingerprint_metadata_df.copy()
if morgan_fingerprint_df["drug"].duplicated().any():
    raise ValueError("Duplicate drug names in Morgan fingerprint table.")
morgan_fingerprint_df = morgan_fingerprint_df.loc[morgan_fingerprint_df["has_fingerprint"]].reset_index(drop=True)
fingerprint_vectors = [fingerprint_to_vector(b) for b in morgan_fingerprint_df["morgan_fingerprint"]]
morgan_fingerprint_tensor = torch.from_numpy(np.vstack(fingerprint_vectors).astype(np.float32))
morgan_drug_to_index = {drug: idx for idx, drug in enumerate(morgan_fingerprint_df["drug"])}

print(
    f"DMSO bundle: {dmso_baseline_expression_tensor.shape}; "
    f"treatment bundle: {treatment_expression_tensor.shape}; "
    f"plates seen: {len(plate_to_index)}."
)

## Plate × Drug Contingency Diagnostic

If a drug lives on a single plate, the adversary's plate label partially overlaps with drug identity for those rows. Document the spread now so we can read training curves with that caveat.

In [ ]:
plates_per_drug = treatment_df.groupby("drug")["plate"].nunique()
drugs_per_plate = treatment_df.groupby("plate")["drug"].nunique()
rows_per_plate = treatment_df["plate"].value_counts().sort_index()

diagnostic_summary_df = pd.DataFrame(
    [
        {"metric": "n_treatment_rows", "value": int(len(treatment_df))},
        {"metric": "n_unique_drugs", "value": int(treatment_df["drug"].nunique())},
        {"metric": "n_unique_plates", "value": int(treatment_df["plate"].nunique())},
        {"metric": "n_unique_cell_lines", "value": int(treatment_df["cell_line"].nunique())},
        {"metric": "plates_per_drug__min", "value": int(plates_per_drug.min())},
        {"metric": "plates_per_drug__median", "value": float(plates_per_drug.median())},
        {"metric": "plates_per_drug__max", "value": int(plates_per_drug.max())},
        {"metric": "drugs_per_plate__min", "value": int(drugs_per_plate.min())},
        {"metric": "drugs_per_plate__median", "value": float(drugs_per_plate.median())},
        {"metric": "drugs_per_plate__max", "value": int(drugs_per_plate.max())},
        {"metric": "n_drugs_on_single_plate", "value": int((plates_per_drug == 1).sum())},
    ]
)

rows_per_plate_df = (
    rows_per_plate.rename("n_rows").reset_index().rename(columns={"index": "plate"})
)

if (plates_per_drug == 1).any():
    n_single = int((plates_per_drug == 1).sum())
    print(
        f"WARNING: {n_single} drugs appear on a single plate across all cell lines. "
        "Plate adversary will partially overlap with drug identity for those rows."
    )
else:
    print("Every drug spans 2+ plates -> the plate signal is not collinear with drug identity.")

display(diagnostic_summary_df)
display(rows_per_plate_df)
display(plates_per_drug.describe().to_frame("plates_per_drug"))

## Persist Plate-Resolved Tensor Artifacts

In [ ]:
dmso_baseline_bundle = {
    "expressions": dmso_baseline_expression_tensor,
    "gene_ids": gene_ids,
    "cell_lines": dmso_baseline_df["cell_line"].tolist(),
    "file_names": dmso_baseline_df["file_name"].tolist(),
    "cell_line_to_index": dmso_cell_line_to_index,
    "source_counts": torch.tensor(dmso_baseline_df["source_count"].to_numpy(), dtype=torch.int64),
    "total_n_cells_used": torch.tensor(dmso_baseline_df["total_n_cells_used"].to_numpy(), dtype=torch.int64),
}

treatment_expression_bundle = {
    "expressions": treatment_expression_tensor,
    "gene_ids": treatment_landmark_gene_ids,
    "feature_space": LANDMARK_FEATURE_SPACE,
    "requested_landmark_gene_ids": requested_landmark_gene_ids,
    "available_landmark_gene_ids": available_landmark_gene_ids,
    "missing_landmark_gene_ids": missing_landmark_gene_ids,
    "condition_keys": treatment_df["condition_key"].tolist(),
    "cell_lines": treatment_df["cell_line"].tolist(),
    "file_names": treatment_df["file_name"].tolist(),
    "drug_names": treatment_df["drug"].tolist(),
    "concentrations": torch.tensor(treatment_df["concentration"].to_numpy(), dtype=torch.float32),
    "concentration_units": treatment_df["concentration_unit"].tolist(),
    "plates": treatment_df["plate"].tolist(),
    "plate_to_index": plate_to_index,
    "source_counts": torch.tensor(treatment_df["source_count"].to_numpy(), dtype=torch.int64),
    "n_cells_used": torch.tensor(treatment_df["n_cells_used"].to_numpy(), dtype=torch.int64),
    "condition_key_to_index": treatment_condition_key_to_index,
}

morgan_fingerprint_bundle = {
    "fingerprints": morgan_fingerprint_tensor,
    "drug_names": morgan_fingerprint_df["drug"].tolist(),
    "pubchem_cids": morgan_fingerprint_df["pubchem_cid"].tolist(),
    "drug_to_index": morgan_drug_to_index,
    "has_fingerprint": torch.tensor(morgan_fingerprint_df["has_fingerprint"].to_numpy(), dtype=torch.bool),
}

saved_artifacts_df = None
if save_tensor_artifacts:
    tensor_artifacts_dir.mkdir(parents=True, exist_ok=True)
    dmso_path = tensor_artifacts_dir / "dmso_baselines.pt"
    treatment_path = tensor_artifacts_dir / "treatment_expressions.pt"
    fingerprint_path = tensor_artifacts_dir / "morgan_fingerprints.pt"
    plate_diagnostic_path = tensor_artifacts_dir / "plate_drug_contingency.csv"

    torch.save(dmso_baseline_bundle, dmso_path)
    torch.save(treatment_expression_bundle, treatment_path)
    torch.save(morgan_fingerprint_bundle, fingerprint_path)

    contingency_df = (
        treatment_df.groupby(["plate", "drug"])
        .size()
        .reset_index(name="n_rows")
        .sort_values(["plate", "drug"], ignore_index=True)
    )
    contingency_df.to_csv(plate_diagnostic_path, index=False)

    saved_artifacts_df = pd.DataFrame(
        [
            {"artifact": "dmso_baselines", "path": str(dmso_path), "size_mb": round(dmso_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "treatment_expressions", "path": str(treatment_path), "size_mb": round(treatment_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "morgan_fingerprints", "path": str(fingerprint_path), "size_mb": round(fingerprint_path.stat().st_size / (1024 ** 2), 3)},
            {"artifact": "plate_drug_contingency", "path": str(plate_diagnostic_path), "size_mb": round(plate_diagnostic_path.stat().st_size / (1024 ** 2), 3)},
        ]
    )
    print(f"Saved plate-resolved tensor artifacts to {tensor_artifacts_dir}")
    display(saved_artifacts_df)
else:
    print(f"save_tensor_artifacts is False; bundles built in memory only ({tensor_artifacts_dir} not written).")